In [17]:
#CELL1

import numpy as np
import pandas as pd
from pathlib import Path

STUDENT_ID = "tupm-26-3597"
SEED_DIGIT = int(STUDENT_ID[-1])
NUM_FEATURES = 5 + (SEED_DIGIT % 3)

print("Seed Digit:", SEED_DIGIT)
print("Features to Select:", NUM_FEATURES)



Seed Digit: 7
Features to Select: 6


In [18]:
from pathlib import Path

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

print("Measurement file found:", measurement_path.is_file())

Measurement file found: True


In [19]:
#CELL2

raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float,
    ndmin=2
)

print("Dataset Shape:", raw_measurements.shape)
print("Number of Records:", raw_measurements.shape[0])
print("Number of Features:", raw_measurements.shape[1])
print("Data Type:", raw_measurements.dtype)

Dataset Shape: (1567, 590)
Number of Records: 1567
Number of Features: 590
Data Type: float64


In [20]:
measurements = raw_measurements.copy()
measurements[~np.isfinite(measurements)] = np.nan
missing_counts = np.isnan(measurements).sum(axis=0)
valid_counts = np.sum(~np.isnan(measurements), axis=0)
all_missing = valid_counts == 0
constant_features = np.zeros(measurements.shape[1], dtype=bool)

for i in range(measurements.shape[1]):
    valid_values = measurements[:, i][~np.isnan(measurements[:, i])]

    if len(valid_values) > 0:
        constant_features[i] = np.min(valid_values) == np.max(valid_values)

usable_features = np.where(~all_missing & ~constant_features)[0]

print("Total Features:", measurements.shape[1])
print("Features with Missing Values:", np.sum(missing_counts > 0))
print("Completely Missing Features:", np.sum(all_missing))
print("Constant Features:", np.sum(constant_features))
print("Usable Features:", len(usable_features))

Total Features: 590
Features with Missing Values: 538
Completely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [21]:
assert len(usable_features) >= NUM_FEATURES, "Not enough usable features."
rng = np.random.default_rng(SEED_DIGIT)

selected_features = rng.choice(
    usable_features,
    size=NUM_FEATURES,
    replace=False
)
selected_features = np.sort(selected_features)

print("Selected Feature Indices:", selected_features)
print("Number of Selected Features:", len(selected_features))

Selected Feature Indices: [321 348 389 447 539 559]
Number of Selected Features: 6


In [22]:
selected_data = measurements[:, selected_features]
feature_means = np.nanmean(selected_data, axis=0)
feature_medians = np.nanmedian(selected_data, axis=0)
feature_stds = np.nanstd(selected_data, axis=0, ddof=1)



missing_percentages = (
    np.isnan(selected_data).sum(axis=0)
    / selected_data.shape[0]
) * 100

print("Feature Means:", feature_means)
print("Feature Medians:", feature_medians)
print("Sample Standard Deviations:", feature_stds)
print("Missing Percentages:", missing_percentages)

Feature Means: [2.26372739e+00 2.47057032e-02 1.33835354e-03 2.81895338e-01
 3.03423489e+00 3.25708174e-01]
Feature Medians: [2.07765e+00 2.26000e-02 3.00000e-04 2.79700e-01 3.05480e+00 2.90900e-01]
Sample Standard Deviations: [2.11699449 0.01186248 0.011816   0.08646076 1.25291304 0.20139187]
Missing Percentages: [0.06381621 1.53158902 0.         0.06381621 0.51052967 0.06381621]


In [23]:
absolute_deviations = np.abs(
    selected_data - feature_medians
)

MAD = np.nanmedian(absolute_deviations, axis=0)
robust_scores = np.full(selected_data.shape, np.nan)

for i in range(NUM_FEATURES):

    valid = ~np.isnan(selected_data[:, i])

    if MAD[i] > 0:
        robust_scores[valid, i] = (
            0.6745 *
            (selected_data[valid, i] - feature_medians[i])
            / MAD[i]
        )

    else:
        robust_scores[valid, i] = np.where(
            selected_data[valid, i] == feature_medians[i],
            0,
            np.inf
        )

print("Median Absolute Deviations:", MAD)
print("Robust Scores Shape:", robust_scores.shape)
print("Robust Standardization Completed.")

Median Absolute Deviations: [4.5540e-01 4.7000e-03 1.0000e-04 5.6800e-02 1.0167e+00 1.4500e-01]
Robust Scores Shape: (1567, 6)
Robust Standardization Completed.


In [24]:
flagged_mask = np.abs(robust_scores) > 3.5
flagged_counts = flagged_mask.sum(axis=0)
flagged_percentages = (
    flagged_counts / selected_data.shape[0]
) * 100
total_flagged_observations = np.any(
    flagged_mask, axis=1
).sum()
print("Flagged Counts per Feature:", flagged_counts)
print("Flagged Percentages:", flagged_percentages)
print("Total Flagged Observations:", total_flagged_observations)

Flagged Counts per Feature: [ 20  50 184   0   0   1]
Flagged Percentages: [ 1.27632419  3.19081047 11.74218251  0.          0.          0.06381621]
Total Flagged Observations: 247


In [25]:
feature_names = [
    f"Feature_{i + 1}" for i in selected_features
]

summary = pd.DataFrame({
    "Feature": feature_names,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_Std": feature_stds,
    "Missing_Percentage": missing_percentages,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

summary = summary.sort_values(
    by="Flagged_Percentage",
    ascending=False
)

summary.insert(
    0,
    "Rank",
    range(1, len(summary) + 1)
)

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "measurement_quality.csv"
summary.to_csv(output_path, index=False)

print("FEATURE QUALITY SUMMARY:")
display(summary)
print("CSV Saved:", output_path.resolve())

FEATURE QUALITY SUMMARY:


,Rank,Feature,Mean,Median,Sample_Std,Missing_Percentage,Flagged_Count,Flagged_Percentage
2,1,Feature_390,0.001338,0.00030,0.011816,0.000000,184,11.742183
1,2,Feature_349,0.024706,0.02260,0.011862,1.531589,50,3.190810
0,3,Feature_322,2.263727,2.07765,2.116994,0.063816,20,1.276324
5,4,Feature_560,0.325708,0.29090,0.201392,0.063816,1,0.063816
3,5,Feature_448,0.281895,0.27970,0.086461,0.063816,0,0.000000
4,6,Feature_540,3.034235,3.05480,1.252913,0.510530,0,0.000000


CSV Saved: C:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab7_NumPy_Pandas_Parras\output\measurement_quality.csv


In [26]:
highest_missing_index = np.argmax(missing_percentages)

highest_flagged_index = np.argmax(flagged_percentages)

verification = np.array_equal(
    flagged_mask,
    np.abs(robust_scores) > 3.5
)


print("\n1. Selected Features:")
print(selected_features.tolist())

print("\n2. Highest Missing Percentage:")
print(f"{missing_percentages[highest_missing_index]:.4f}%")

print("\n3. Feature with Highest Flagged Percentage:")
print(f"Feature_{selected_features[highest_flagged_index] + 1}")

print("\n4. Total Flagged Observations:")
print(total_flagged_observations)

print("\n5. Vectorized Results Verified:")
print("YES" if verification else "NO")




1. Selected Features:
[321, 348, 389, 447, 539, 559]

2. Highest Missing Percentage:
1.5316%

3. Feature with Highest Flagged Percentage:
Feature_390

4. Total Flagged Observations:
247

5. Vectorized Results Verified:
YES


In [27]:
highest_missing_index = np.argmax(missing_percentages)

highest_flagged_index = np.argmax(flagged_percentages)

verification = np.array_equal(
    flagged_mask,
    np.abs(robust_scores) > 3.5
)


print("Selected Features:", (selected_features.tolist()))
print("Highest Missing Percentage:", f"{missing_percentages[highest_missing_index]:.4f}%")
print("Feature with Highest Flagged Percentage:", f"Feature_{selected_features[highest_flagged_index] + 1}")
print("Total Flagged Observations:", total_flagged_observations)
print("Vectorized Results Verified:", "YES" if verification else "NO")




Selected Features: [321, 348, 389, 447, 539, 559]
Highest Missing Percentage: 1.5316%
Feature with Highest Flagged Percentage: Feature_390
Total Flagged Observations: 247
Vectorized Results Verified: YES
